In [1]:
import re
import pandas as pd
import json
import openai
import time
from google.colab import userdata
import os
from openai import OpenAI

In [2]:
from google.colab import files
uploaded=files.upload()

Saving pan12-sexual-predator-identification-training-corpus-2012-05-01.txt to pan12-sexual-predator-identification-training-corpus-2012-05-01.txt


In [3]:
with open('/content/pan12-sexual-predator-identification-training-corpus-2012-05-01.txt', 'r', encoding='utf-8') as f:
    context = f.read()
context[:100]

'<conversations>\n  <conversation id="e621da5de598c9321a1d505ea95e6a2d">\n    <message line="1">\n      '

In [4]:
import xml.etree.ElementTree as ET
import json

def xml_to_json(xml_str):
    root = ET.fromstring(xml_str)
    conversations = []

    for conv in root.findall('conversation'):
        conv_id = conv.attrib.get('id', '')
        messages = []

        for msg in conv.findall('message'):
            messages.append({
                'line': msg.attrib.get('line', ''),
                'author': msg.findtext('author', ''),
                'time': msg.findtext('time', ''),
                'text': msg.findtext('text', '')
                    .replace("&apos;", "'")
                    .replace("&quot;", '"')
                    .replace("&amp;", "&")
            })

        conversations.append({
            'conversation_id': conv_id,
            'messages': messages
        })

    return conversations

In [5]:
# 변환 수행
context_json = xml_to_json(context)

In [6]:
context_json

[{'conversation_id': 'e621da5de598c9321a1d505ea95e6a2d',
  'messages': [{'line': '1',
    'author': '97964e7a9e8eb9cf78f2e4d7b2ff34c7',
    'time': '03:20',
    'text': 'Hola.'},
   {'line': '2',
    'author': '0158d0d6781fc4d493f243d4caa49747',
    'time': '03:20',
    'text': 'hi.'},
   {'line': '3',
    'author': '0158d0d6781fc4d493f243d4caa49747',
    'time': '03:20',
    'text': 'whats up?'},
   {'line': '4',
    'author': '97964e7a9e8eb9cf78f2e4d7b2ff34c7',
    'time': '03:20',
    'text': 'not a ton.'},
   {'line': '5',
    'author': '97964e7a9e8eb9cf78f2e4d7b2ff34c7',
    'time': '03:20',
    'text': 'you?'},
   {'line': '6',
    'author': '0158d0d6781fc4d493f243d4caa49747',
    'time': '03:20',
    'text': 'same.  being lazy.  M or f?'},
   {'line': '7',
    'author': '97964e7a9e8eb9cf78f2e4d7b2ff34c7',
    'time': '03:20',
    'text': 'F.'},
   {'line': '8',
    'author': '97964e7a9e8eb9cf78f2e4d7b2ff34c7',
    'time': '03:21',
    'text': "Ditto, I've done absolutely nothing

In [7]:
!pip install openai --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 720.5/720.5 kB 10.0 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.81.0
    Uninstalling openai-1.81.0:
      Successfully uninstalled openai-1.81.0


In [8]:
system_prompt = """
You are a professional translator specializing in informal online chat messages.

Your job is to translate short, informal English messages into simple, natural Korean, just like how two Koreans would casually chat online.

Key rules:
- Focus on the overall meaning and tone rather than word-for-word translation.
- Maintain the casual tone, slang, abbreviations, or emojis where appropriate.
- Do not add or omit any information that wasn't in the original.
- Do not localize proper nouns (e.g., "New York" is 뉴욕, but "Netflix" stays Netflix).
- Keep it short, natural, and fluent. Don’t overtranslate.
"""

def gpt_translate(text):
    if not text.strip():
        return ""

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt.strip()},
                {"role": "user", "content": text}
            ],
            temperature=0.2,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"[❌ 오류] {text[:30]}... → {e}")
        return text  # 번역 실패 시 원문 그대로

In [9]:
os.environ['OPENAI_API_KEY'] = userdata.get('Translation_Key')

# 🔑 OpenAI API 키
client = OpenAI(
  api_key=os.environ['OPENAI_API_KEY'],
)

i = 0

for conv in context_json:
    if i == 1:
        break
    i += 1
    for msg in conv["messages"]:
        original_text = msg.get("text", "")
        translated = gpt_translate(original_text)
        msg["text"] = translated
        print(f"원문 : {original_text} -> 번역 : {translated}")


In [10]:
context_json[0]

{'conversation_id': 'e621da5de598c9321a1d505ea95e6a2d',
 'messages': [{'line': '1',
   'author': '97964e7a9e8eb9cf78f2e4d7b2ff34c7',
   'time': '03:20',
   'text': '안녕!'},
  {'line': '2',
   'author': '0158d0d6781fc4d493f243d4caa49747',
   'time': '03:20',
   'text': '안녕!'},
  {'line': '3',
   'author': '0158d0d6781fc4d493f243d4caa49747',
   'time': '03:20',
   'text': '뭐해?'},
  {'line': '4',
   'author': '97964e7a9e8eb9cf78f2e4d7b2ff34c7',
   'time': '03:20',
   'text': '별로 없어.'},
  {'line': '5',
   'author': '97964e7a9e8eb9cf78f2e4d7b2ff34c7',
   'time': '03:20',
   'text': '너는?'},
  {'line': '6',
   'author': '0158d0d6781fc4d493f243d4caa49747',
   'time': '03:20',
   'text': '나도 똑같아. 게으름 피우고 있어. 남자야 여자야?'},
  {'line': '7',
   'author': '97964e7a9e8eb9cf78f2e4d7b2ff34c7',
   'time': '03:20',
   'text': 'F.'},
  {'line': '8',
   'author': '97964e7a9e8eb9cf78f2e4d7b2ff34c7',
   'time': '03:21',
   'text': '나도! 나도 하루 종일 Hulu에서만 놀았어. 😂'},
  {'line': '9',
   'author': '0158d0d6781fc4d493f

In [11]:
msg["text"]

'안녕!'